# Veridelta: Quickstart Guide

Veridelta is a high-performance semantic diffing engine. Standard binary equality checks often fail in production due to floating-point noise, string formatting variance, or schema evolution. Veridelta allows you to define 'logical equality' through declarative configuration.

### 1. The Problem: Physical vs. Logical Equality
In this scenario, a data migration has introduced minor floating-point jitter (rounding errors) and inconsistent whitespace, though the data remains logically identical.

In [4]:
import polars as pl

# Legacy system data
src_df = pl.DataFrame({
    "id": [1, 2, 3],
    "amount": [100.0, 250.50, 300.0],
    "status": ["active", "inactive", "pending"]
})

# Modern warehouse data with jitter and padding
tgt_df = pl.DataFrame({
    "id": [1, 2, 3],
    "amount": [100.04, 250.49, 300.0],
    "status": [" ACTIVE ", "INACTIVE", " PENDING "]
})

print(f"Standard Equality Check: {src_df.equals(tgt_df)}")

Standard Equality Check: False


### 2. Semantic Configuration
We use `DiffConfig` to define the comparison boundaries. This configuration is separated from the I/O logic to allow for both in-memory and file-based workflows.

In [5]:
from veridelta import ColumnRule, DiffConfig, DiffEngine

config = DiffConfig(
    # Define columns used for row-level alignment (joins)
    primary_keys=["id"],

    # Global absolute tolerance: allow up to 0.05 units of difference for all numeric columns
    default_absolute_tolerance=0.05,

    # Global whitespace policy: strip leading and trailing spaces from all string columns
    default_whitespace_mode="both",

    # Granular overrides for specific columns
    rules=[
        ColumnRule(
            column_names=["status"],
            case_insensitive=True  # Ensure 'active' matches ' ACTIVE '
        )
    ]
)

# The DiffEngine consumes the config and the aligned DataFrames
engine = DiffEngine(config, src_df, tgt_df)
summary = engine.run()

print("VERIDELTA RESULTS:")
print(f"Match Status:  {summary.is_match}")
print(f"Changed Rows:  {summary.changed_count}")

VERIDELTA RESULTS:
Match Status:  True
Changed Rows:  0


### 3. Enterprise Ingestion & Semantic Drift

In production data pipelines, drift commonly occurs during ETL transformations. This example ingests a subset of the **NYC Yellow Taxi Trip Records (January 2024)**, using the first 1,000 rows to simulate a migration scenario. The target system introduces floating-point noise and inconsistent string padding to represent realistic downstream transformation variance.

The raw data artifacts used in this example are available in the [Veridelta documentation assets directory](https://github.com/Veridelta/veridelta/tree/main/docs/assets/data).

In [ ]:
from veridelta import DataIngestor, SourceConfig

# Remote dataset URI: NYC Taxi sample (January 2024, first 1000 rows)
DATA_URL = "https://raw.githubusercontent.com/Veridelta/veridelta/main/docs/assets/data/sample_taxi_data.parquet"

# 1. Configuration: Define logical equality boundaries
config = DiffConfig(
    primary_keys=["id"],
    default_absolute_tolerance=0.01,  # Global absolute tolerance for numeric comparisons
    default_whitespace_mode="both",   # Global stripping of leading/trailing whitespace
    rules=[
        ColumnRule(
            column_names=["store_and_fwd_flag"],
            case_insensitive=True     # Normalize casing for boolean indicators
        )
    ]
)

# 2. Ingestion: Fetch source of truth via DataIngestor
source_io = SourceConfig(path=DATA_URL, format="parquet")
ingestor = DataIngestor(config, source_io, source_io)
src_df, _ = ingestor.get_dataframes()

# 3. Simulate Drift: Programmatically introduce "jitters" to the target dataset
# This represents common rounding noise and inconsistent padding in modern warehouses.
tgt_df = src_df.with_columns([
    (pl.col("fare_amount") + 0.002).alias("fare_amount"),
    (pl.col("store_and_fwd_flag") + " ").alias("store_and_fwd_flag")
])

# 4. Engine Execution: High-performance semantic comparison
engine = DiffEngine(config, src_df, tgt_df)
summary = engine.run()

# 5. Result Verification
print(f"{' VERIDELTA EXECUTION SUMMARY ':=^50}")
print(f"Match Status:     {'SUCCESS' if summary.is_match else 'FAILED'}")
print(f"Total Rows:       {summary.total_rows_source:,}")
print(f"Semantic Diffs:   {summary.changed_count:,}")
print("=" * 50)

Execution Flow:
1. DataIngestor fetches raw datasets via Polars.
2. DataIngestor normalizes headers and aligns schemas.
3. DiffEngine performs the multi-threaded semantic comparison.


### Summary

Veridelta provides a rigorous framework for data validation in high-throughput environments.

* **Zero-Import Configuration**: Semantic modes use literal strings for reduced boilerplate.
* **Strictly Typed Architecture**: Built on Pydantic V2 for high-speed validation of configuration logic.
* **Computational Efficiency**: Direct integration with Polars ensures comparison logic scales with your hardware.